# 02 — Train adaptive retrieval planner

Fine-tune Qwen2.5-1.5B-Instruct on empirically selected ONCE/SPLIT plans using `adaptive_rag.planner.adaptive`. Run after notebook 01 has exported `planner_train.jsonl`. **Run on a Colab GPU** (Runtime → Change runtime type). The notebook orchestrates the package functions; training logic stays in the package.

## 1. Configuration
Edit these values before running. Set `GITHUB_REPO` to your public GitHub repository or install from a checked-out local directory. Training outputs are stored on Google Drive so they survive Colab runtime resets.
The base model (`SLM_MODEL`) and random seed (`SEED`) are imported from `adaptive_rag.config` after package installation; do not duplicate them in the configuration cell.


In [ ]:
from pathlib import Path

GITHUB_REPO = "RanaShoaaib/Adaptive-RAG"  # e.g. user/adaptive-rag
GITHUB_REF = "main"  # preferably a commit SHA for reproducibility
INSTALL_FROM_GITHUB = True

MOUNT_DRIVE = True
DRIVE_ROOT = "/content/drive/MyDrive/Adaptive-RAG"
SFT_PATH = f"{DRIVE_ROOT}/subsets/planner_training/planner_train.jsonl"
OUTPUT_DIR = f"{DRIVE_ROOT}/results/adaptive_planner_v1"

USE_4BIT = True  # QLoRA, recommended on Colab GPU
MAX_LENGTH = 768
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
EPOCHS = 3.0
LEARNING_RATE = 2e-4
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
VALIDATION_FRACTION = 0.2  # split labeled training examples only
RUN_TRAINING = False  # set True only after inspecting the dataset
RUN_INFERENCE_SMOKE_TEST = True

## 2. Install the project and training dependencies
`pip install` in Colab affects only the Colab runtime, not your local Conda environment. Re-run after a fresh runtime. Your GitHub repository must include the latest `adaptive.py` and `pyproject.toml`.

In [ ]:
import subprocess, sys
if INSTALL_FROM_GITHUB:
    url = f"git+https://github.com/{GITHUB_REPO}.git@{GITHUB_REF}"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", f"adaptive-rag[training] @ {url}"])
else:
    # Use when repository is already present at /content/Adaptive-RAG
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", "/content/Adaptive-RAG[training]"])

## 3. GPU and persistent storage

In [ ]:
from adaptive_rag.config import SLM_MODEL, SEED
BASE_MODEL = SLM_MODEL

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab before training")
print("GPU:", torch.cuda.get_device_name(0))

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

from pathlib import Path
sft_path = Path(SFT_PATH)
output_dir = Path(OUTPUT_DIR)
assert sft_path.is_file(), f"Missing SFT data: {sft_path}. Copy notebook 01 output here."
output_dir.mkdir(parents=True, exist_ok=True)
print("Data:", sft_path)
print("Output:", output_dir)

## 4. Validate examples and create a reproducible training/validation split
The SFT file should contain **only training-subset questions**. The 200-question external validation and test subsets must not be mixed into this file. Here, `VALIDATION_FRACTION` reserves part of the labeled training subset for fine-tuning loss monitoring.

In [ ]:
from collections import Counter
from adaptive_rag.planner.adaptive import load_sft_examples, split_sft_examples

records = load_sft_examples(sft_path)
train_records, val_records = split_sft_examples(
    records, validation_fraction=VALIDATION_FRACTION, seed=SEED
)
print(f"Total labeled: {len(records)} | SFT train: {len(train_records)} | SFT validation: {len(val_records)}")
print("All labels:", dict(Counter(r["target"]["strategy"] for r in records)))
print("Train labels:", dict(Counter(r["target"]["strategy"] for r in train_records)))
print("Val labels:", dict(Counter(r["target"]["strategy"] for r in val_records)))
print("Example question:", records[0]["question"])
print("Example target:", records[0]["target"])
assert not ({r["question_id"] for r in train_records} & {r["question_id"] for r in val_records})

## 5. Save training manifest
This records the settings used for the run. Avoid overwriting an earlier output directory when comparing experiments.

In [ ]:
import json
manifest = {
    "github_repo": GITHUB_REPO if INSTALL_FROM_GITHUB else None,
    "github_ref": GITHUB_REF if INSTALL_FROM_GITHUB else None,
    "sft_path": str(sft_path),
    "output_dir": str(output_dir),
    "base_model": BASE_MODEL,
    "use_4bit": USE_4BIT,
    "max_length": MAX_LENGTH,
    "rank": LORA_RANK,
    "alpha": LORA_ALPHA,
    "dropout": LORA_DROPOUT,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "validation_fraction": VALIDATION_FRACTION,
    "seed": SEED,
    "n_total": len(records),
    "n_train": len(train_records),
    "n_val": len(val_records),
}
manifest_path = output_dir / "notebook_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Saved:", manifest_path)

## 6. Train QLoRA/LoRA adapter
Set `RUN_TRAINING = True` in the configuration cell when ready. `train_planner()` saves epoch checkpoints, the final adapter, tokenizer, and `training_config.json` to `OUTPUT_DIR`. Note: the current module starts a new run; it does not expose `resume_from_checkpoint`.

In [ ]:
from adaptive_rag.planner.adaptive import train_planner

trainer = model = tokenizer = None
if RUN_TRAINING:
    trainer, model, tokenizer = train_planner(
        train_records=train_records,
        val_records=val_records,
        output_dir=output_dir,
        model_name=BASE_MODEL,
        use_4bit=USE_4BIT,
        max_length=MAX_LENGTH,
        rank=LORA_RANK,
        alpha=LORA_ALPHA,
        dropout=LORA_DROPOUT,
        epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        seed=SEED,
    )
    print("Saved adapter:", output_dir / "adapter")
else:
    print("Training skipped. Set RUN_TRAINING=True and rerun this cell to start.")

## 7. Review loss history and export metrics

In [ ]:
import pandas as pd
if trainer is not None:
    history = pd.DataFrame(trainer.state.log_history)
    history_path = output_dir / "training_history.csv"
    history.to_csv(history_path, index=False)
    display(history.tail(15))
    print("Saved:", history_path)
else:
    print("Run training first to view logs.")

In [ ]:
if trainer is not None:
    import matplotlib.pyplot as plt
    history = pd.DataFrame(trainer.state.log_history)
    fig, ax = plt.subplots(figsize=(7, 4))
    for key, label in [("loss", "Training loss"), ("eval_loss", "Validation loss")]:
        part = history.dropna(subset=[key]) if key in history else pd.DataFrame()
        if not part.empty:
            ax.plot(part["step"], part[key], marker="o", label=label)
    ax.set(xlabel="Optimizer step", ylabel="Loss", title="Planner fine-tuning loss")
    ax.legend()
    plt.show()

## 8. Adapter smoke test (same quantization as evaluation)
Load the saved adapter and generate one plan. This tests the adapter-loading path and JSON schema; it is **not** a substitute for the held-out evaluation in notebook 03.

In [ ]:
from adaptive_rag.planner.adaptive import load_adaptive_planner, generate_adaptive_query
adapter_dir = output_dir / "adapter"
if RUN_INFERENCE_SMOKE_TEST and adapter_dir.is_dir():
    if model is not None:
        del model, trainer
        import gc
        gc.collect()
        torch.cuda.empty_cache()
    inference_model, inference_tokenizer = load_adaptive_planner(
        adapter_dir, base_model_name=BASE_MODEL, use_4bit=USE_4BIT
    )
    example = "Which university did the author of the novel attend?"
    plan = generate_adaptive_query(inference_model, inference_tokenizer, example)
    print(json.dumps(plan, indent=2))
elif RUN_INFERENCE_SMOKE_TEST:
    print("Adapter not found yet:", adapter_dir)

## Outputs and next step
Expected artifacts: `adapter/`, `checkpoints/`, `training_config.json`, `notebook_manifest.json`, and `training_history.csv`. Use `adapter/` in `03_evaluate_planner.ipynb`. Keep the independent validation/test questions untouched for model comparison.

**Compatibility note:** This notebook uses the interfaces in the uploaded `adaptive_rag/planner/adaptive.py`. Runtime compatibility with the exact Colab Transformers/PEFT versions still needs testing; if the `torch_dtype` or `TrainingArguments` API differs, adjust the package module rather than duplicating training logic here.